# 学习RAG：逐步测试配置
## 带有增强评估的教育性端到端管道

此笔记本旨在作为一个学习项目，以了解不同设置如何影响检索增强生成（RAG）系统。我们将使用 **Nebius AI API** 步骤构建和测试一个管道。

**我们将学习的内容：**
*   文本分块（`chunk_size`、`chunk_overlap`）如何影响RAG系统检索的内容。
*   检索的文档数量（`top_k`）如何影响提供给LLM的上下文。
*   三种常见RAG策略（简单、查询重写、重排序）之间的区别。
*   如何使用LLM（如Nebius AI）通过多个指标自动评估生成答案的质量：与真实答案的**忠实度**、**相关性**和**语义相似度**。
*   如何将这些指标合并为一个平均分数，以便于比较。

我们将重点关注每一步执行的*原因*，并通过详细解释和带注释的代码清晰地观察结果。

### 目录

1. **设置：安装库**：获取必要的工具。
2. **设置：导入库**：将工具引入我们的工作空间。
3. **配置：设置实验**：定义API详细信息、模型、评估提示和要测试的参数。
4. **输入数据：知识源和问题**：定义RAG系统将从中学习的文档以及我们将要提问的问题。
5. **核心组件：文本分块函数**：创建一个将文档分解为较小部分的函数。
6. **核心组件：连接到Nebius AI**：建立连接以使用Nebius模型。
7. **核心组件：余弦相似度函数**：创建一个用于测量文本之间语义相似度的函数。
8. **实验：迭代配置**：测试不同设置的主要循环。
    *   8.1 处理分块配置（分块、嵌入、索引）
    *   8.2 测试`top_k`值的RAG策略
    *   8.3 运行和评估单一RAG策略（包括相似度）
9. **分析：回顾结果**：使用Pandas组织和显示结果。
10. **结论：我们学到了什么？**：反思发现和潜在的下一步行动。

### 1. 设置：安装库

首先，我们需要安装此笔记本所需的Python软件包。
- `openai`：与Nebius API交互（该接口与OpenAI兼容）。
- `pandas`：用于创建和管理数据表（数据框）。
- `numpy`：用于数值操作，特别是向量（嵌入）操作。
- `faiss-cpu`：用于在向量上进行高效相似性搜索（检索部分）。
- `ipywidgets`、`tqdm`：用于在Jupyter中显示进度条。
- `scikit-learn`：用于计算余弦相似度。

In [1]:
# Install libraries (run this cell only once if needed)
# !pip install openai pandas numpy faiss-cpu ipywidgets tqdm scikit-learn

**注意！** 安装完成后，您可能需要**重启内核**（或运行时），以便Jupyter/Colab识别新安装的软件包。您可以在菜单中查找此选项（例如，“内核”->“重启内核...”或“运行时”->“重启运行时”）。

### 2. 设置：导入库

完成库的安装后，我们将它们引入Python环境以使用其功能。

In [2]:
import os                     # For accessing environment variables (like API keys)
import time                   # For timing operations
import re                     # For regular expressions (text cleaning)
import warnings               # For controlling warning messages
import itertools              # For creating parameter combinations easily
import getpass                # For securely prompting for API keys if not set

import numpy as np            # Numerical library for vector operations
import pandas as pd           # Data manipulation library for tables (DataFrames)
import faiss                  # Library for fast vector similarity search
from openai import OpenAI     # Client library for Nebius API interaction
from tqdm.notebook import tqdm # Library for displaying progress bars
from sklearn.metrics.pairwise import cosine_similarity # For calculating similarity score

# Configure display options for Pandas DataFrames for better readability
pd.set_option('display.max_colwidth', 150) # Show more text content in table cells
pd.set_option('display.max_rows', 100)     # Display more rows in tables
warnings.filterwarnings('ignore', category=FutureWarning) # Suppress specific non-critical warnings

print("Libraries imported successfully!")

Libraries imported successfully!


### 3. 配置：设置实验

在这里，我们直接将实验的所有设置和参数定义为Python变量。这使得在一处查看和修改配置变得容易。

**关键配置领域：**
*   **Nebius API详细信息**：连接到Nebius AI的凭证和模型标识符。
*   **LLM设置**：控制语言模型在生成答案期间行为的参数（例如，用于创意性的`temperature`）。
*   **评估提示**：当LLM充当忠实度和相关性评估者时，给予LLM的具体指令（提示）。
*   **调优参数**：我们想要系统测试的分块大小、重叠和检索`top_k`的不同值。
*   **重排序设置**：模拟重排序策略的配置。

In [ ]:
# --- NebiusAI API Configuration ---
# It's best practice to store API keys as environment variables rather than hardcoding them.
# Provide your actual key here or set it as an environment variable
NEBIUS_API_KEY = os.getenv('NEBIUS_API_KEY', None)  # Load API key from environment variable
if NEBIUS_API_KEY is None:
    print("Warning: NEBIUS_API_KEY not set. Please set it in your environment variables or provide it directly in the code.") 
NEBIUS_BASE_URL = "https://api.studio.nebius.com/v1/" 
NEBIUS_EMBEDDING_MODEL = "BAAI/bge-multilingual-gemma2"  # Model for converting text to vector embeddings
NEBIUS_GENERATION_MODEL = "deepseek-ai/DeepSeek-V3"    # LLM for generating the final answers
NEBIUS_EVALUATION_MODEL = "deepseek-ai/DeepSeek-V3"    # LLM used for evaluating the generated answers

# --- Text Generation Parameters (for RAG answer generation) ---
GENERATION_TEMPERATURE = 0.1  # Lower values (e.g., 0.1-0.3) make output more focused and deterministic, good for fact-based answers.
GENERATION_MAX_TOKENS = 400   # Maximum number of tokens (roughly words/sub-words) in the generated answer.
GENERATION_TOP_P = 0.9        # Nucleus sampling parameter (alternative to temperature, usually fine at default).

# --- Evaluation Prompts (Instructions for the Evaluator LLM) ---
# Faithfulness: Does the answer stay true to the provided context?
FAITHFULNESS_PROMPT = """
System: You are an objective evaluator. Evaluate the faithfulness of the AI Response compared to the True Answer, considering only the information present in the True Answer as the ground truth.
Faithfulness measures how accurately the AI response reflects the information in the True Answer, without adding unsupported facts or contradicting it.
Score STRICTLY using a float between 0.0 and 1.0, based on this scale:
- 0.0: Completely unfaithful, contradicts or fabricates information.
- 0.1-0.4: Low faithfulness with significant inaccuracies or unsupported claims.
- 0.5-0.6: Partially faithful but with noticeable inaccuracies or omissions.
- 0.7-0.8: Mostly faithful with only minor inaccuracies or phrasing differences.
- 0.9: Very faithful, slight wording differences but semantically aligned.
- 1.0: Completely faithful, accurately reflects the True Answer.
Respond ONLY with the numerical score.

User:
Query: {question}
AI Response: {response}
True Answer: {true_answer}
Score:"""

# Relevancy: Does the answer directly address the user's query?
RELEVANCY_PROMPT = """
System: You are an objective evaluator. Evaluate the relevance of the AI Response to the specific User Query.
Relevancy measures how well the response directly answers the user's question, avoiding unnecessary or off-topic information.
Score STRICTLY using a float between 0.0 and 1.0, based on this scale:
- 0.0: Not relevant at all.
- 0.1-0.4: Low relevance, addresses a different topic or misses the core question.
- 0.5-0.6: Partially relevant, answers only a part of the query or is tangentially related.
- 0.7-0.8: Mostly relevant, addresses the main aspects of the query but might include minor irrelevant details.
- 0.9: Highly relevant, directly answers the query with minimal extra information.
- 1.0: Completely relevant, directly and fully answers the exact question asked.
Respond ONLY with the numerical score.

User:
Query: {question}
AI Response: {response}
Score:"""

# --- Parameters to Tune (The experimental variables) ---
CHUNK_SIZES_TO_TEST = [150, 250]    # List of chunk sizes (in words) to experiment with.
CHUNK_OVERLAPS_TO_TEST = [30, 50]   # List of chunk overlaps (in words) to experiment with.
RETRIEVAL_TOP_K_TO_TEST = [3, 5]   # List of 'k' values (number of chunks to retrieve) to test.

# --- Reranking Configuration (Only used for the Rerank strategy) ---
RERANK_RETRIEVAL_MULTIPLIER = 3 # For simulated reranking: retrieve K * multiplier chunks initially.

# --- Validate API Key --- 
print("--- Configuration Check --- ")
print(f"Attempting to load Nebius API Key from environment variable 'NEBIUS_API_KEY'...")
if not NEBIUS_API_KEY:
    print("Nebius API Key not found in environment variables.")
    # Prompt the user securely if the key is not found.
    NEBIUS_API_KEY = getpass.getpass("Please enter your Nebius API Key: ")
else:
    print("Nebius API Key loaded successfully from environment variable.")

# Print a summary of key settings for verification
print(f"Models: Embed='{NEBIUS_EMBEDDING_MODEL}', Gen='{NEBIUS_GENERATION_MODEL}', Eval='{NEBIUS_EVALUATION_MODEL}'")
print(f"Chunk Sizes to Test: {CHUNK_SIZES_TO_TEST}")
print(f"Overlaps to Test: {CHUNK_OVERLAPS_TO_TEST}")
print(f"Top-K Values to Test: {RETRIEVAL_TOP_K_TO_TEST}")
print(f"Generation Temp: {GENERATION_TEMPERATURE}, Max Tokens: {GENERATION_MAX_TOKENS}")
print("Configuration ready.")
print("-" * 25)

--- Configuration Check --- 
Attempting to load Nebius API Key from environment variable 'NEBIUS_API_KEY'...
Nebius API Key loaded successfully from environment variable.
Models: Embed='BAAI/bge-multilingual-gemma2', Gen='deepseek-ai/DeepSeek-V3', Eval='deepseek-ai/DeepSeek-V3'
Chunk Sizes to Test: [150, 250]
Overlaps to Test: [30, 50]
Top-K Values to Test: [3, 5]
Generation Temp: 0.1, Max Tokens: 400
Configuration ready.
-------------------------


### 4. 输入数据：知识源和问题

每个RAG系统都需要一个知识库来提取信息。在这里，我们定义：
*   `corpus_texts`：一个字符串列表，每个字符串是一个包含信息的文档（在此案例中，关于可再生能源的信息）。
*   `test_query`：我们希望RAG系统使用 `corpus_texts` 来回答的具体问题。
*   `true_answer_for_query`：一个精心编写的“真实答案”，它仅基于 `corpus_texts` 中的信息。这对于准确评估**忠实度**和**语义相似度**至关重要。

In [4]:
# Our knowledge base: A list of text documents about renewable energy
corpus_texts = [
    "Solar power uses PV panels or CSP systems. PV converts sunlight directly to electricity. CSP uses mirrors to heat fluid driving a turbine. It's clean but varies with weather/time. Storage (batteries) is key for consistency.", # Doc 0
    "Wind energy uses turbines in wind farms. It's sustainable with low operating costs. Wind speed varies, siting can be challenging (visual/noise). Offshore wind is stronger and more consistent.", # Doc 1
    "Hydropower uses moving water, often via dams spinning turbines. Reliable, large-scale power with flood control/water storage benefits. Big dams harm ecosystems and displace communities. Run-of-river is smaller, less disruptive.", # Doc 2
    "Geothermal energy uses Earth's heat via steam/hot water for turbines. Consistent 24/7 power, small footprint. High initial drilling costs, sites are geographically limited.", # Doc 3
    "Biomass energy from organic matter (wood, crops, waste). Burned directly or converted to biofuels. Uses waste, provides dispatchable power. Requires sustainable sourcing. Combustion releases emissions (carbon-neutral if balanced by regrowth)." # Doc 4
]

# The question we will ask the RAG system
test_query = "Compare the consistency and environmental impact of solar power versus hydropower."

# !!! CRITICAL: The 'True Answer' MUST be derivable ONLY from the corpus_texts above !!!
# This is our ground truth for evaluation.
true_answer_for_query = "Solar power's consistency varies with weather and time of day, requiring storage like batteries. Hydropower is generally reliable, but large dams have significant environmental impacts on ecosystems and communities, unlike solar power's primary impact being land use for panels."

print(f"Loaded {len(corpus_texts)} documents into our corpus.")
print(f"Test Query: '{test_query}'")
print(f"Reference (True) Answer for evaluation: '{true_answer_for_query}'")
print("Input data is ready.")
print("-" * 25)

Loaded 5 documents into our corpus.
Test Query: 'Compare the consistency and environmental impact of solar power versus hydropower.'
Reference (True) Answer for evaluation: 'Solar power's consistency varies with weather and time of day, requiring storage like batteries. Hydropower is generally reliable, but large dams have significant environmental impacts on ecosystems and communities, unlike solar power's primary impact being land use for panels.'
Input data is ready.
-------------------------


### 5. 核心组件：文本分块函数

LLM 和嵌入模型在一次处理的文本量上有局限。此外，检索在搜索较小、专注的文本片段时效果最佳，而非在整个大型文档上进行。

**分块** 是将大型文档拆分为较小、可能重叠的片段的过程。

- **`chunk_size`**：确定每个分块的大致大小（在此以单词数计）。
- **`chunk_overlap`**：指定一个分块末尾的多少单词也应包含在下一个分块的开头。这有助于防止相关跨分块边界的信息丢失。

我们定义一个基于单词数进行拆分的 `chunk_text` 函数。

In [5]:
def chunk_text(text, chunk_size, chunk_overlap):
    """Splits a single text document into overlapping chunks based on word count.

    Args:
        text (str): The input text to be chunked.
        chunk_size (int): The target number of words per chunk.
        chunk_overlap (int): The number of words to overlap between consecutive chunks.

    Returns:
        list[str]: A list of text chunks.
    """
    words = text.split()      # Split the text into a list of individual words
    total_words = len(words) # Calculate the total number of words in the text
    chunks = []             # Initialize an empty list to store the generated chunks
    start_index = 0         # Initialize the starting word index for the first chunk

    # --- Input Validation ---
    # Ensure chunk_size is a positive integer.
    if not isinstance(chunk_size, int) or chunk_size <= 0:
        print(f"  Warning: Invalid chunk_size ({chunk_size}). Must be a positive integer. Returning the whole text as one chunk.")
        return [text]
    # Ensure chunk_overlap is a non-negative integer smaller than chunk_size.
    if not isinstance(chunk_overlap, int) or chunk_overlap < 0:
        print(f"  Warning: Invalid chunk_overlap ({chunk_overlap}). Must be a non-negative integer. Setting overlap to 0.")
        chunk_overlap = 0
    if chunk_overlap >= chunk_size:
        # If overlap is too large, adjust it to a reasonable fraction (e.g., 1/3) of chunk_size
        # This prevents infinite loops or nonsensical chunking.
        adjusted_overlap = chunk_size // 3
        print(f"  Warning: chunk_overlap ({chunk_overlap}) >= chunk_size ({chunk_size}). Adjusting overlap to {adjusted_overlap}.")
        chunk_overlap = adjusted_overlap

    # --- Chunking Loop ---
    # Continue chunking as long as the start_index is within the bounds of the text
    while start_index < total_words:
        # Determine the end index for the current chunk.
        # It's the minimum of (start + chunk_size) and the total number of words.
        end_index = min(start_index + chunk_size, total_words)
        
        # Extract the words for the current chunk and join them back into a single string.
        current_chunk_text = " ".join(words[start_index:end_index])
        chunks.append(current_chunk_text) # Add the generated chunk to the list
        
        # Calculate the starting index for the *next* chunk.
        # Move forward by (chunk_size - chunk_overlap) words.
        next_start_index = start_index + chunk_size - chunk_overlap
        
        # --- Safety Checks ---
        # Check 1: Prevent infinite loops if overlap causes no progress.
        # This can happen if chunk_size is very small or overlap is very large relative to chunk_size.
        if next_start_index <= start_index:
            if end_index == total_words: # If we are already at the end, we can safely break.
                break
            else: 
                # Force progress by moving forward by at least one word.
                print(f"  Warning: Chunking logic stuck (start={start_index}, next_start={next_start_index}). Forcing progress.")
                next_start_index = start_index + 1 
                
        # Check 2: If the calculated next start index is already at or beyond the total number of words, we are done.
        if next_start_index >= total_words:
            break
            
        # Move the start_index to the calculated position for the next iteration.
        start_index = next_start_index
        
    return chunks # Return the complete list of text chunks

# --- Quick Test ---
# Test the function with the first document and sample parameters.
print("Defining the 'chunk_text' function.")
sample_chunk_size = 150
sample_overlap = 30
sample_chunks = chunk_text(corpus_texts[0], sample_chunk_size, sample_overlap) 
print(f"Test chunking on first doc (size={sample_chunk_size} words, overlap={sample_overlap} words): Created {len(sample_chunks)} chunks.")
if sample_chunks: # Only print if chunks were created
    print(f"First sample chunk:\n'{sample_chunks[0]}'")
print("-" * 25)

Defining the 'chunk_text' function.
Test chunking on first doc (size=150 words, overlap=30 words): Created 1 chunks.
First sample chunk:
'Solar power uses PV panels or CSP systems. PV converts sunlight directly to electricity. CSP uses mirrors to heat fluid driving a turbine. It's clean but varies with weather/time. Storage (batteries) is key for consistency.'
-------------------------


### 6. 核心组件：连接到Nebius AI

要使用Nebius AI模型（用于嵌入、生成、评估），我们需要建立与Nebius API的连接。我们使用`openai` Python库，它提供了与Nebius等OpenAI兼容API交互的便捷方式。

我们实例化一个`OpenAI`客户端对象，提供我们的API密钥和特定的Nebius API端点URL。

In [6]:
client = None # Initialize client variable to None globally

print("Attempting to initialize the Nebius AI client...")
try:
    # Check if the API key is actually available before creating the client
    if not NEBIUS_API_KEY:
        raise ValueError("Nebius API Key is missing. Cannot initialize client.")
        
    # Create the OpenAI client object, configured for the Nebius API.
    client = OpenAI(
        api_key=NEBIUS_API_KEY,     # Pass the API key loaded earlier
        base_url=NEBIUS_BASE_URL  # Specify the Nebius API endpoint
    )
    
    # Optional: Add a quick test call to verify the client connection,
    # e.g., listing models (if supported and desired). This might incur costs.
    # try:
    #     client.models.list() 
    #     print("Client connection verified by listing models.")
    # except Exception as test_e:
    #     print(f"Warning: Could not verify client connection with test call: {test_e}")
    
    print("Nebius AI client initialized successfully. Ready to make API calls.")
    
except Exception as e:
    # Catch any errors during client initialization (e.g., invalid key, network issues)
    print(f"Error initializing Nebius AI client: {e}")
    print("!!! Execution cannot proceed without a valid client. Please check your API key and network connection. !!!")
    # Setting client back to None to prevent further attempts if initialization failed
    client = None 

print("Client setup step complete.")
print("-" * 25)

Attempting to initialize the Nebius AI client...
Nebius AI client initialized successfully. Ready to make API calls.
Client setup step complete.
-------------------------


### 7. 核心组件：余弦相似度函数

为了评估生成的答案与真实答案在语义上的相似度，我们使用**余弦相似度**。该指标衡量两个向量（在我们的情况下，是两个答案的嵌入向量）之间角度的余弦值。

- **1** 的分数表示向量方向相同（最大相似度）。
- **0** 的分数表示向量正交（无相似度）。
- **-1** 的分数表示向量方向相反（最大不相似度）。

对于文本嵌入，分数通常在 0 到 1 之间，较高的值表示语义相似度更大。

我们定义了一个函数 `calculate_cosine_similarity`，它接受两个文本字符串，使用Nebius客户端生成它们的嵌入向量，并返回它们的余弦相似度分数。

In [7]:
def calculate_cosine_similarity(text1, text2, client, embedding_model):
    """Calculates cosine similarity between the embeddings of two texts.

    Args:
        text1 (str): The first text string.
        text2 (str): The second text string.
        client (OpenAI): The initialized Nebius AI client.
        embedding_model (str): The name of the embedding model to use.

    Returns:
        float: The cosine similarity score (between 0.0 and 1.0), or 0.0 if an error occurs.
    """
    if not client:
        print("  Error: Nebius client not available for similarity calculation.")
        return 0.0
    if not text1 or not text2:
        # Handle cases where one or both texts might be empty or None
        return 0.0
        
    try:
        # Generate embeddings for both texts in a single API call if possible
        response = client.embeddings.create(model=embedding_model, input=[text1, text2])
        
        # Extract the embedding vectors
        embedding1 = np.array(response.data[0].embedding)
        embedding2 = np.array(response.data[1].embedding)
        
        # Reshape vectors to be 2D arrays as expected by cosine_similarity
        embedding1 = embedding1.reshape(1, -1)
        embedding2 = embedding2.reshape(1, -1)
        
        # Calculate cosine similarity using scikit-learn
        # cosine_similarity returns a 2D array, e.g., [[similarity]], so we extract the value.
        similarity_score = cosine_similarity(embedding1, embedding2)[0][0]
        
        # Clamp the score between 0.0 and 1.0 for safety/consistency
        return max(0.0, min(1.0, similarity_score))
        
    except Exception as e:
        print(f"  Error calculating cosine similarity: {e}")
        return 0.0 # Return 0.0 in case of any API or calculation errors

# --- Quick Test ---
print("Defining the 'calculate_cosine_similarity' function.")
if client: # Only run test if client is initialized
    test_sim = calculate_cosine_similarity("apple", "orange", client, NEBIUS_EMBEDDING_MODEL)
    print(f"Testing similarity function: Similarity between 'apple' and 'orange' = {test_sim:.2f}")
else:
    print("Skipping similarity function test as Nebius client is not initialized.")
print("-" * 25)

Defining the 'calculate_cosine_similarity' function.
Testing similarity function: Similarity between 'apple' and 'orange' = 0.77
-------------------------


### 8. 实验：迭代配置

本节包含主要的实验循环。我们将系统地迭代之前定义的所有调优参数组合（`CHUNK_SIZES_TO_TEST`、`CHUNK_OVERLAPS_TO_TEST`、`RETRIEVAL_TOP_K_TO_TEST`）。

**每个参数组合的工作流程：**

1.  **准备数据（分块/嵌入/索引 - 步骤 8.1）：**
    *   **检查是否需要重新计算：** 如果 `chunk_size` 或 `chunk_overlap` 与上一次迭代不同，则需要重新处理语料库。
    *   **分块：** 使用当前的 `chunk_size` 和 `chunk_overlap` 通过 `chunk_text` 函数将 `corpus_texts` 中的所有文档拆分。
    *   **嵌入：** 使用指定的Nebius嵌入模型（`NEBIUS_EMBEDDING_MODEL`）将每个文本块转换为数值向量（嵌入）。我们为提高效率以批次进行此操作。
    *   **索引：** 从生成的嵌入向量构建FAISS索引（`IndexFlatL2`）。FAISS可进行非常快速的搜索，以找到与查询嵌入最相似的块的嵌入向量。
    *   *优化：* 如果分块设置未变，我们从上一次迭代中重复使用现有的块、嵌入和索引，以节省时间和API调用。

2.  **测试RAG策略（步骤 8.2）：**
    *   对于当前的 `top_k` 值，运行每个定义的RAG策略：
        *   **简单RAG：** 基于与原始查询的相似性检索 `top_k` 个块。
        *   **查询重写RAG：** 首先，要求LLM重写原始查询，使其更适合向量搜索。然后，基于与重写查询的相似性检索 `top_k` 个块。
        *   **重排序RAG（模拟）：** 最初检索更多块（`top_k * RERANK_RETRIEVAL_MULTIPLIER`）。然后，通过从这个更大的初始集合中简单地取前 `top_k` 个结果来模拟重排序。（实际实现会使用更复杂的重排序模型）。

3.  **评估和存储结果（在 `run_and_evaluate` 中的步骤 8.3）：**
    *   对于每个策略运行：
        *   **检索：** 使用FAISS索引找到相关的块索引。
        *   **生成：** 构建一个包含检索到的块作为上下文和原始 `test_query` 的提示。将其发送到Nebius生成模型（`NEBIUS_GENERATION_MODEL`）以获取最终答案。
        *   **评估（忠实度）：** 使用LLM评估器（`NEBIUS_EVALUATION_MODEL`）和 `FAITHFULNESS_PROMPT` 来评分生成的答案与 `true_answer_for_query` 的一致性。
        *   **评估（相关性）：** 使用LLM评估器和 `RELEVANCY_PROMPT` 来评分生成的答案对 `test_query` 的针对性。
        *   **评估（相似度）：** 使用我们的 `calculate_cosine_similarity` 函数来获取生成的答案与 `true_answer_for_query` 之间的语义相似度分数。
        *   **计算平均分数：** 计算忠实度、相关性和相似度分数的平均值。
        *   **记录：** 存储所有参数（`chunk_size`、`overlap`、`top_k`、`strategy`），检索到的索引，重写查询（如适用），生成的答案，各个分数，平均分数以及此次特定运行的执行时间。

我们使用 `tqdm` 为迭代参数组合的外层循环显示进度条。

In [8]:
# List to store the detailed results from each experimental run
all_results = []

# --- Cache variables for Chunking/Embedding/Indexing --- 
# These variables help us avoid redundant computations when only 'top_k' changes.
last_chunk_size = -1      # Stores the chunk_size used in the previous iteration
last_overlap = -1         # Stores the chunk_overlap used in the previous iteration
current_index = None      # Holds the active FAISS index
current_chunks = []       # Holds the list of text chunks for the active settings
current_embeddings = None # Holds the numpy array of embeddings for the active chunks

# Check if the Nebius client was initialized successfully before starting
if not client:
    print("STOPPING: Nebius AI client is not initialized. Cannot run experiment.")
else:
    print("=== Starting RAG Experiment Loop ===\n")
    
    # Create all possible combinations of the tuning parameters
    param_combinations = list(itertools.product(
        CHUNK_SIZES_TO_TEST,
        CHUNK_OVERLAPS_TO_TEST,
        RETRIEVAL_TOP_K_TO_TEST
    ))
    
    print(f"Total parameter combinations to test: {len(param_combinations)}")
    
    # --- Main Loop --- 
    # Iterate through each combination (chunk_size, chunk_overlap, top_k)
    # Use tqdm to display a progress bar.
    for chunk_size, chunk_overlap, top_k in tqdm(param_combinations, desc="Testing Configurations"):
        
        # --- 8.1 Processing a Chunking Configuration --- 
        # Check if chunk settings have changed, requiring re-processing.
        if chunk_size != last_chunk_size or chunk_overlap != last_overlap:
            # Uncomment the line below for more verbose logging during execution
            # print(f"\n--- Processing New Chunk Config: Size={chunk_size}, Overlap={chunk_overlap} ---")
            
            # Update cache variables
            last_chunk_size, last_overlap = chunk_size, chunk_overlap
            # Reset index, chunks, and embeddings for the new configuration
            current_index = None 
            current_chunks = []
            current_embeddings = None
            
            # --- 8.1a: Chunking --- 
            # Apply the chunk_text function to each document in the corpus
            try:
                # print("  Chunking documents...") # Uncomment for verbose logging
                temp_chunks = []
                for doc_index, doc in enumerate(corpus_texts):
                    doc_chunks = chunk_text(doc, chunk_size, chunk_overlap)
                    if not doc_chunks:
                         print(f"  Warning: No chunks created for document {doc_index} with size={chunk_size}, overlap={chunk_overlap}. Skipping document.")
                         continue
                    temp_chunks.extend(doc_chunks)
                
                current_chunks = temp_chunks
                if not current_chunks:
                    # If no chunks were created at all (e.g., due to invalid settings or empty corpus)
                    raise ValueError("No chunks were created for the current configuration.")
                # print(f"    Created {len(current_chunks)} chunks total.") # Uncomment for verbose logging
            except Exception as e:
                 print(f"    ERROR during chunking for Size={chunk_size}, Overlap={chunk_overlap}: {e}. Skipping this configuration.")
                 last_chunk_size, last_overlap = -1, -1 # Reset cache state
                 continue # Move to the next parameter combination
            
            # --- 8.1b: Embedding --- 
            # Generate embeddings for all chunks using the Nebius API.
            # print("  Generating embeddings...") # Uncomment for verbose logging
            try:
                batch_size = 32 # Process chunks in batches to avoid overwhelming the API or hitting limits.
                temp_embeddings = [] # Temporary list to store embedding vectors
                
                # Loop through chunks in batches
                for i in range(0, len(current_chunks), batch_size):
                    batch_texts = current_chunks[i : min(i + batch_size, len(current_chunks))]
                    # Make the API call to Nebius for the current batch
                    response = client.embeddings.create(model=NEBIUS_EMBEDDING_MODEL, input=batch_texts)
                    # Extract the embedding vectors from the API response
                    batch_embeddings = [item.embedding for item in response.data]
                    temp_embeddings.extend(batch_embeddings)
                    time.sleep(0.05) # Add a small delay between batches to be polite to the API endpoint.
                
                # Convert the list of embeddings into a single NumPy array
                current_embeddings = np.array(temp_embeddings)
                # Basic validation check on the embeddings array
                if current_embeddings.ndim != 2 or current_embeddings.shape[0] != len(current_chunks):
                    raise ValueError(f"Embeddings array shape mismatch. Expected ({len(current_chunks)}, dim), Got {current_embeddings.shape}")
                # print(f"    Generated {current_embeddings.shape[0]} embeddings (Dimension: {current_embeddings.shape[1]}).") # Uncomment for verbose logging

            except Exception as e:
                print(f"    ERROR generating embeddings for Size={chunk_size}, Overlap={chunk_overlap}: {e}. Skipping this chunk config.")
                # Reset cache variables to indicate failure for this chunk setting
                last_chunk_size, last_overlap = -1, -1 
                current_chunks = []
                current_embeddings = None
                continue # Skip to the next parameter combination
                
            # --- 8.1c: Indexing --- 
            # Build a FAISS index for efficient similarity search.
            # print("  Building FAISS search index...") # Uncomment for verbose logging
            try:
                embedding_dim = current_embeddings.shape[1] # Get the dimensionality of the embeddings
                # We use IndexFlatL2, which performs exact search using L2 (Euclidean) distance.
                # For high-dimensional vectors from modern embedding models, cosine similarity often works better,
                # but FAISS's IndexFlatIP (Inner Product) is closely related. For normalized embeddings (like many BGE models),
                # L2 distance and Inner Product/Cosine Similarity ranking are equivalent.
                current_index = faiss.IndexFlatL2(embedding_dim)
                # Add the chunk embeddings to the index. FAISS requires float32 data type.
                current_index.add(current_embeddings.astype('float32'))
                
                if current_index.ntotal == 0:
                     raise ValueError("FAISS index is empty after adding vectors. No vectors were added.")
                # print(f"    FAISS index ready with {current_index.ntotal} vectors.") # Uncomment for verbose logging
            except Exception as e:
                print(f"    ERROR building FAISS index for Size={chunk_size}, Overlap={chunk_overlap}: {e}. Skipping this chunk config.")
                # Reset variables to indicate failure
                last_chunk_size, last_overlap = -1, -1
                current_index = None
                current_embeddings = None
                current_chunks = []
                continue # Skip to the next parameter combination
        
        # --- 8.2 Testing RAG Strategies for the Current Top-K --- 
        # If we reach this point, we have a valid index and chunks for the current chunk_size/overlap.
        
        # Check if the index and chunks are actually available (safety check)
        if current_index is None or not current_chunks:
            print(f"    WARNING: Index or chunks not available for Size={chunk_size}, Overlap={chunk_overlap}. Skipping Top-K={top_k} test.")
            continue
            
        # --- 8.3 Running & Evaluating a Single RAG Strategy --- 
        # Define a nested function to perform the core RAG steps (retrieve, generate, evaluate)
        # This avoids code repetition for each strategy.
        def run_and_evaluate(strategy_name, query_to_use, k_retrieve, use_simulated_rerank=False):
            # print(f"    Starting: {strategy_name} (k={k_retrieve}) ...") # Uncomment for verbose logging
            run_start_time = time.time() # Record start time for timing the run
            
            # Initialize a dictionary to store results for this specific run
            result = {
                'chunk_size': chunk_size, 'overlap': chunk_overlap, 'top_k': k_retrieve, 
                'strategy': strategy_name,
                'retrieved_indices': [], 'rewritten_query': None, 'answer': 'Error: Execution Failed',
                'faithfulness': 0.0, 'relevancy': 0.0, 'similarity_score': 0.0, 'avg_score': 0.0, 
                'time_sec': 0.0
            }
            # Store the rewritten query if applicable
            if strategy_name == "Query Rewrite RAG": 
                result['rewritten_query'] = query_to_use

            try:
                # --- Retrieval Step --- 
                k_for_search = k_retrieve # Number of chunks to retrieve initially
                if use_simulated_rerank:
                    # For simulated rerank, retrieve more candidates initially
                    k_for_search = k_retrieve * RERANK_RETRIEVAL_MULTIPLIER
                    # print(f"      Rerank: Retrieving initial {k_for_search} candidates.") # Uncomment for verbose logging
                
                # 1. Embed the query (original or rewritten)
                query_embedding_response = client.embeddings.create(model=NEBIUS_EMBEDDING_MODEL, input=[query_to_use])
                query_embedding = query_embedding_response.data[0].embedding
                query_vector = np.array([query_embedding]).astype('float32') # FAISS needs float32 numpy array
                
                # 2. Perform the search in the FAISS index
                # Ensure k is not greater than the total number of items in the index
                actual_k = min(k_for_search, current_index.ntotal)
                if actual_k == 0:
                    raise ValueError("Index is empty or k_for_search is zero, cannot search.")
                
                # `current_index.search` returns distances and indices of the nearest neighbors
                distances, indices = current_index.search(query_vector, actual_k)
                
                # 3. Process retrieved indices
                # Indices can contain -1 if fewer than 'actual_k' vectors are found (shouldn't happen with IndexFlatL2 unless k > ntotal)
                retrieved_indices_all = indices[0]
                valid_indices = retrieved_indices_all[retrieved_indices_all != -1].tolist()
                
                # 4. Apply simulated reranking (if applicable)
                # In this simulation, we just take the top 'k_retrieve' results from the initially retrieved set.
                # A real reranker would re-score these 'k_for_search' candidates based on relevance to the query.
                if use_simulated_rerank:
                    final_indices = valid_indices[:k_retrieve]
                    # print(f"      Rerank: Selected top {len(final_indices)} indices after simulated rerank.") # Uncomment for verbose logging
                else:
                    final_indices = valid_indices # Use all valid retrieved indices up to k_retrieve
                
                result['retrieved_indices'] = final_indices
                
                # 5. Get the actual text chunks corresponding to the final indices
                retrieved_chunks = [current_chunks[i] for i in final_indices]
                
                # Handle case where no chunks were retrieved (should be rare with valid indices)
                if not retrieved_chunks:
                    print(f"      Warning: No relevant chunks found for {strategy_name} (C={chunk_size}, O={chunk_overlap}, K={k_retrieve}). Setting answer to indicate this.")
                    result['answer'] = "No relevant context found in the documents based on the query."
                    # Keep scores at 0.0 as no answer was generated from context
                else:
                    # --- Generation Step --- 
                    # Combine the retrieved chunks into a single context string
                    context_str = "\n\n".join(retrieved_chunks)
                    
                    # Define the system prompt for the generation LLM
                    sys_prompt_gen = "You are a helpful AI assistant. Answer the user's query based strictly on the provided context. If the context doesn't contain the answer, state that clearly. Be concise."
                    
                    # Construct the user prompt including the context and the *original* query
                    # It's important to use the original query here for generating the final answer, even if a rewritten query was used for retrieval.
                    user_prompt_gen = f"Context:\n------\n{context_str}\n------\n\nQuery: {test_query}\n\nAnswer:"
                    
                    # Make the API call to the Nebius generation model
                    gen_response = client.chat.completions.create(
                        model=NEBIUS_GENERATION_MODEL, 
                        messages=[
                            {"role": "system", "content": sys_prompt_gen},
                            {"role": "user", "content": user_prompt_gen}
                        ],
                        temperature=GENERATION_TEMPERATURE,
                        max_tokens=GENERATION_MAX_TOKENS,
                        top_p=GENERATION_TOP_P
                    )
                    # Extract the generated text answer
                    generated_answer = gen_response.choices[0].message.content.strip()
                    result['answer'] = generated_answer
                    # Optional: print a snippet of the generated answer
                    # print(f"      Generated Answer: {generated_answer[:100].replace('\n', ' ')}...") 

                    # --- Evaluation Step --- 
                    # Evaluate the generated answer using Faithfulness, Relevancy, and Similarity
                    # print(f"      Evaluating answer... (Faithfulness, Relevancy, Similarity)") # Uncomment for verbose logging
                    
                    # Prepare parameters for evaluation calls (use low temperature for deterministic scoring)
                    eval_params = {'model': NEBIUS_EVALUATION_MODEL, 'temperature': 0.0, 'max_tokens': 10}
                    
                    # 1. Faithfulness Evaluation Call
                    prompt_f = FAITHFULNESS_PROMPT.format(question=test_query, response=generated_answer, true_answer=true_answer_for_query)
                    try:
                        resp_f = client.chat.completions.create(messages=[{"role": "user", "content": prompt_f}], **eval_params)
                        # Attempt to parse the score, clamp between 0.0 and 1.0
                        result['faithfulness'] = max(0.0, min(1.0, float(resp_f.choices[0].message.content.strip())))
                    except Exception as eval_e:
                        print(f"      Warning: Faithfulness score parsing error for {strategy_name} - {eval_e}. Score set to 0.0")
                        result['faithfulness'] = 0.0

                    # 2. Relevancy Evaluation Call
                    prompt_r = RELEVANCY_PROMPT.format(question=test_query, response=generated_answer)
                    try:
                        resp_r = client.chat.completions.create(messages=[{"role": "user", "content": prompt_r}], **eval_params)
                        # Attempt to parse the score, clamp between 0.0 and 1.0
                        result['relevancy'] = max(0.0, min(1.0, float(resp_r.choices[0].message.content.strip())))
                    except Exception as eval_e:
                        print(f"      Warning: Relevancy score parsing error for {strategy_name} - {eval_e}. Score set to 0.0")
                        result['relevancy'] = 0.0
                    
                    # 3. Similarity Score Calculation
                    result['similarity_score'] = calculate_cosine_similarity(
                        generated_answer, 
                        true_answer_for_query, 
                        client, 
                        NEBIUS_EMBEDDING_MODEL
                    )
                    
                    # 4. Calculate Average Score (Faithfulness, Relevancy, Similarity)
                    result['avg_score'] = (result['faithfulness'] + result['relevancy'] + result['similarity_score']) / 3.0
            
            except Exception as e:
                # Catch any unexpected errors during the retrieve/generate/evaluate process
                error_message = f"ERROR during {strategy_name} (C={chunk_size}, O={chunk_overlap}, K={k_retrieve}): {str(e)[:200]}..."
                print(f"    {error_message}")
                result['answer'] = error_message # Store the error in the answer field
                # Ensure scores remain at their default error state (0.0)
                result['faithfulness'] = 0.0
                result['relevancy'] = 0.0
                result['similarity_score'] = 0.0
                result['avg_score'] = 0.0
            
            # Record the total time taken for this run
            run_end_time = time.time()
            result['time_sec'] = run_end_time - run_start_time
            
            # Print a summary line for this run (useful for monitoring progress)
            print(f"    Finished: {strategy_name} (C={chunk_size}, O={chunk_overlap}, K={k_retrieve}). AvgScore={result['avg_score']:.2f}, Time={result['time_sec']:.2f}s")
            return result
        # --- End of run_and_evaluate nested function ---

        # --- Execute the RAG Strategies using the run_and_evaluate function --- 
        
        # Strategy 1: Simple RAG (Use original query for retrieval)
        result_simple = run_and_evaluate("Simple RAG", test_query, top_k)
        all_results.append(result_simple)

        # Strategy 2: Query Rewrite RAG 
        rewritten_q = test_query # Default to original query if rewrite fails
        try:
             # print("    Attempting query rewrite for Rewrite RAG...") # Uncomment for verbose logging
             # Define prompts for the query rewriting task
             sys_prompt_rw = "You are an expert query optimizer. Rewrite the user's query to be ideal for vector database retrieval. Focus on key entities, concepts, and relationships. Remove conversational fluff. Output ONLY the rewritten query text."
             user_prompt_rw = f"Original Query: {test_query}\n\nRewritten Query:"
             
             # Call the LLM to rewrite the query
             resp_rw = client.chat.completions.create(
                 model=NEBIUS_GENERATION_MODEL, # Can use the generation model for this task too
                 messages=[
                     {"role": "system", "content": sys_prompt_rw},
                     {"role": "user", "content": user_prompt_rw}
                 ],
                 temperature=0.1, # Low temp for focused rewrite
                 max_tokens=100, 
                 top_p=0.9
             )
             # Clean up the LLM's response to get just the query text
             candidate_q = resp_rw.choices[0].message.content.strip()
             # Remove potential prefixes like "Rewritten Query:" or "Query:"
             candidate_q = re.sub(r'^(rewritten query:|query:)\s*', '', candidate_q, flags=re.IGNORECASE).strip('"')
             
             # Use the rewritten query only if it's reasonably different and not too short
             if candidate_q and len(candidate_q) > 5 and candidate_q.lower() != test_query.lower(): 
                 rewritten_q = candidate_q
                 # print(f"      Using rewritten query: '{rewritten_q}'") # Uncomment for verbose logging
             # else: 
                 # print("      Rewrite failed, too short, or same as original. Using original query.") # Uncomment for verbose logging
        except Exception as e:
             print(f"    Warning: Error during query rewrite: {e}. Using original query.")
             rewritten_q = test_query # Fallback to original query on error
             
        # Run evaluation using the (potentially) rewritten query for retrieval
        result_rewrite = run_and_evaluate("Query Rewrite RAG", rewritten_q, top_k)
        all_results.append(result_rewrite)

        # Strategy 3: Rerank RAG (Simulated)
        # Use original query for retrieval, but simulate the reranking process
        result_rerank = run_and_evaluate("Rerank RAG (Simulated)", test_query, top_k, use_simulated_rerank=True)
        all_results.append(result_rerank)

    print("\n=== RAG Experiment Loop Finished ===")
    print("-" * 25)

=== Starting RAG Experiment Loop ===

Total parameter combinations to test: 8


Testing Configurations:   0%|          | 0/8 [00:00<?, ?it/s]

    Finished: Simple RAG (C=150, O=30, K=3). AvgScore=0.89, Time=609.06s
    Finished: Query Rewrite RAG (C=150, O=30, K=3). AvgScore=0.89, Time=10.36s
    Finished: Rerank RAG (Simulated) (C=150, O=30, K=3). AvgScore=0.89, Time=9.53s
    Finished: Simple RAG (C=150, O=30, K=5). AvgScore=0.89, Time=8.40s
    Finished: Query Rewrite RAG (C=150, O=30, K=5). AvgScore=0.89, Time=8.36s
    Finished: Rerank RAG (Simulated) (C=150, O=30, K=5). AvgScore=0.89, Time=8.34s
    Finished: Simple RAG (C=150, O=50, K=3). AvgScore=0.89, Time=9.78s
    Finished: Query Rewrite RAG (C=150, O=50, K=3). AvgScore=0.89, Time=9.68s
    Finished: Rerank RAG (Simulated) (C=150, O=50, K=3). AvgScore=0.89, Time=8.43s
    Finished: Simple RAG (C=150, O=50, K=5). AvgScore=0.89, Time=9.74s
    Finished: Query Rewrite RAG (C=150, O=50, K=5). AvgScore=0.89, Time=9.39s
    Finished: Rerank RAG (Simulated) (C=150, O=50, K=5). AvgScore=0.89, Time=8.53s
    Finished: Simple RAG (C=250, O=30, K=3). AvgScore=0.89, Time=9.36

### 9. 分析：回顾结果

实验循环已完成，`all_results` 包含每次运行的数据，我们将使用Pandas库分析这些结果。

1.  **创建数据框**：将结果字典列表（`all_results`）转换为Pandas数据框，以便于操作和查看。
2.  **排序结果**：按 `avg_score`（忠实度、相关性和相似度的平均值）降序排序，使表现最佳的配置排在前面。
3.  **显示顶级配置**：展示排序后的数据框的前N行，包括关键参数、分数和生成的答案，以便快速识别有前景的设置。
4.  **总结最佳运行**：基于平均分数，打印出表现最佳的单次运行配置的清晰摘要，显示其参数、各个分数、耗时及生成的完整答案。

In [9]:
print("--- Analyzing Experiment Results ---")

# First, check if any results were actually collected
if not all_results:
    print("No results were generated during the experiment. Cannot perform analysis.")
else:
    # Convert the list of result dictionaries into a Pandas DataFrame
    results_df = pd.DataFrame(all_results)
    print(f"Total results collected: {len(results_df)}")
    
    # Sort the DataFrame based on the 'avg_score' column in descending order (best first)
    # Use reset_index(drop=True) to get a clean 0-based index after sorting.
    results_df_sorted = results_df.sort_values(by='avg_score', ascending=False).reset_index(drop=True)
    
    print("\n--- Top 10 Performing Configurations (Sorted by Average Score) ---")
    # Define the columns we want to display in the summary table
    display_cols = [
        'chunk_size', 'overlap', 'top_k', 'strategy', 
        'avg_score', 'faithfulness', 'relevancy', 'similarity_score', # Added similarity
        'time_sec', 
        'answer' # Including the answer helps qualitatively assess the best runs
    ]
    # Filter out any columns that might not exist (e.g., if an error occurred before population)
    display_cols = [col for col in display_cols if col in results_df_sorted.columns]
    
    # Display the head (top 10 rows) of the sorted DataFrame using the selected columns
    # The display() function provides richer output in Jupyter environments.
    display(results_df_sorted[display_cols].head(10))
    
    # --- Summary of the Single Best Run --- 
    print("\n--- Best Configuration Summary ---")
    # Check if the sorted DataFrame is not empty before accessing the first row
    if not results_df_sorted.empty:
        # Get the first row (index 0), which corresponds to the best score after sorting
        best_run = results_df_sorted.iloc[0]
        
        # Print the parameters and results of the best configuration
        print(f"Chunk Size: {best_run.get('chunk_size', 'N/A')} words")
        print(f"Overlap: {best_run.get('overlap', 'N/A')} words")
        print(f"Top-K Retrieved: {best_run.get('top_k', 'N/A')} chunks")
        print(f"Strategy: {best_run.get('strategy', 'N/A')}")
        # Use .get(col, default) for robustness in case a column is missing
        avg_score = best_run.get('avg_score', 0.0)
        faithfulness = best_run.get('faithfulness', 0.0)
        relevancy = best_run.get('relevancy', 0.0)
        similarity = best_run.get('similarity_score', 0.0)
        time_sec = best_run.get('time_sec', 0.0)
        best_answer = best_run.get('answer', 'N/A')
        
        print(f"---> Average Score (Faith+Rel+Sim): {avg_score:.3f}")
        print(f"      (Faithfulness: {faithfulness:.3f}, Relevancy: {relevancy:.3f}, Similarity: {similarity:.3f})")
        print(f"Time Taken: {time_sec:.2f} seconds")
        print(f"\nBest Answer Generated:")
        # Print the full answer generated by the best configuration
        print(best_answer)
    else:
        # Handle the case where no results were successfully processed
        print("Could not determine the best configuration (no valid results found).")
        
print("\n--- Analysis Complete --- ")

--- Analyzing Experiment Results ---
Total results collected: 24

--- Top 10 Performing Configurations (Sorted by Average Score) ---


,chunk_size,overlap,top_k,strategy,avg_score,faithfulness,relevancy,similarity_score,time_sec,answer
0,250,50,3,Simple RAG,0.899417,0.9,1.0,0.798251,8.975824,Solar power and hydropower differ significantly in consistency and environmental impact:\n\n- **Consistency**: \n - **Solar Power**: Inconsisten...
1,250,50,3,Query Rewrite RAG,0.896859,0.9,1.0,0.790578,6.550637,"**Consistency:**\n- **Hydropower** is highly reliable and provides consistent, large-scale power, as it is not dependent on weather conditions onc..."
2,150,30,3,Rerank RAG (Simulated),0.894125,0.9,1.0,0.782374,9.526656,"**Consistency:** \n- **Hydropower** is highly reliable and consistent, providing large-scale power 24/7, as it is not dependent on weather or tim..."
3,150,50,3,Query Rewrite RAG,0.893823,0.9,1.0,0.781468,9.675948,"**Consistency:** \n- **Hydropower** is highly reliable and provides consistent, large-scale power, as it is not dependent on weather conditions o..."
4,150,30,3,Query Rewrite RAG,0.893666,0.9,1.0,0.780997,10.357061,"**Consistency:**\n- **Hydropower** is highly reliable and consistent, providing large-scale power 24/7, as it relies on the continuous flow of wat..."
5,150,50,3,Simple RAG,0.892774,0.9,1.0,0.778321,9.777294,"**Consistency:** \n- **Hydropower** is highly consistent and reliable, providing large-scale power 24/7, especially with dams that store water fo..."
6,250,50,3,Rerank RAG (Simulated),0.891570,0.9,1.0,0.774709,41.228211,"**Consistency:** \n- **Hydropower** is highly reliable and provides consistent, large-scale power, especially with dams that can store water and ..."
7,250,30,3,Query Rewrite RAG,0.890878,0.9,1.0,0.772635,8.359087,"**Consistency:** \n- **Hydropower** is highly reliable and provides consistent, large-scale power, especially with dams that can store water and ..."
8,250,30,5,Simple RAG,0.890867,0.9,1.0,0.772601,8.767287,"**Consistency:** \n- **Solar Power:** Inconsistent due to dependence on weather and daylight. Requires storage solutions (e.g., batteries) for re..."
9,150,50,5,Simple RAG,0.890656,0.9,1.0,0.771967,9.743746,"**Consistency:** \n- **Solar Power:** Inconsistent due to dependence on weather and daylight. Requires storage solutions (e.g., batteries) for re..."



--- Best Configuration Summary ---
Chunk Size: 250 words
Overlap: 50 words
Top-K Retrieved: 3 chunks
Strategy: Simple RAG
---> Average Score (Faith+Rel+Sim): 0.899
      (Faithfulness: 0.900, Relevancy: 1.000, Similarity: 0.798)
Time Taken: 8.98 seconds

Best Answer Generated:
Solar power and hydropower differ significantly in consistency and environmental impact:

- **Consistency**:  
  - **Solar Power**: Inconsistent, as it depends on weather conditions and time of day. Requires storage solutions (like batteries) for reliable supply.  
  - **Hydropower**: Highly consistent, providing large-scale, reliable power 24/7, especially with dams.  

- **Environmental Impact**:  
  - **Solar Power**: Clean with minimal emissions during operation, but manufacturing panels and disposal can have environmental impacts.  
  - **Hydropower**: Large dams can severely harm ecosystems, disrupt fish migration, and displace communities. Run-of-river systems are less disruptive but still impact local en

### 10. 结论：我们学到了什么？

我们已成功构建并执行了一个端到端的管道，用于在Nebius AI平台上实验各种RAG配置，并通过多指标评估其性能。

通过查看结果表和上面的最佳配置摘要，我们可以获得关于*我们选择的语料库、查询和模型*的特定见解。

**反思要点：**

*   **分块的影响：** 是否某个 `chunk_size` 或 `overlap` 倾向于产生更好的平均分数？考虑为什么较小的块可能更好地捕捉特定事实，而较大的块可能提供更多上下文。重叠似乎如何影响结果？
*   **检索数量（`top_k`）：** 增加 `top_k` 如何影响分数？检索更多块是否总是能带来更好的答案，还是有时引入噪声或无关信息，可能降低忠实度或相似度？
*   **策略比较：** “查询重写”或“重排序（模拟）”策略是否在平均分数上相对于“简单RAG”提供了一致的优势？潜在的改进是否足够显著，以证明额外步骤的合理性（例如，重写所需的额外LLM调用，重排序所需的更大初始检索）？
*   **评估指标：**
    *   查看“最佳答案”并将其与 `true_answer_for_query` 进行比较。各个分数（忠实度、相关性、相似度）是否反映了您感知到的质量？
    *   高相似度是否总是与高忠实度相关？答案是否可能相似但不忠实，或者忠实但不相似？
    *   您认为基于LLM的自动评估（忠实度、相关性）与更客观的余弦相似度相比的可靠性如何？基于LLM的评估有哪些潜在限制（例如，对提示措辞的敏感性、模型偏见）？
*   **整体性能：** 是否有任何配置接近完美的平均分数？是什么可能阻止了完美分数（例如，源文档的限制、语言中的固有歧义、不完美的检索）？

**关键收获：** 优化RAG系统是一个迭代过程。最佳配置通常高度依赖于特定数据集、用户查询的性质、选定的嵌入和LLM模型以及评估标准。像本笔记本中遵循的系统实验过程对于找到特定用例表现良好的设置至关重要。

**潜在的下一步行动和进一步探索：**

*   **扩展测试参数：** 尝试更广泛的 `chunk_size`、`overlap` 和 `top_k` 值。
*   **不同查询：** 使用不同类型的查询（例如，基于事实的、比较性的、总结性的）测试相同配置，以查看性能如何变化。
*   **更大/不同的语料库：** 使用更广泛或特定领域的知识库。
*   **实施真正的重排序：** 用专用的交叉编码器重排序模型（例如，来自Hugging Face Transformers或Cohere Rerank）替换模拟重排序，以根据相关性重新评分最初检索到的文档。
*   **替代模型：** 尝试不同的Nebius AI模型用于嵌入、生成或评估，以查看其影响。
*   **高级分块：** 探索更复杂的分块策略（例如，递归字符拆分、语义分块）。
*   **人工评估：** 以人工判断补充自动化指标，以更细致地了解答案质量。